# AutoGPT

Implementation of https://github.com/Significant-Gravitas/Auto-GPT but with LangChain primitives (LLMs, PromptTemplates, VectorStores, Embeddings, Tools)

## Set up tools

We'll set up an AutoGPT with a search tool, and write-file tool, and a read-file tool

In [1]:
from langchain.agents import Tool
from langchain_community.tools.file_management.read import ReadFileTool
from langchain_community.tools.file_management.write import WriteFileTool
from langchain_community.utilities import SerpAPIWrapper
import os
os.environ["SERPAPI_API_KEY"] = "ae10ffac84691f03575e355cfb1edbef5d010deb999b09735105bd97e227e8b3"
# os.environ["OPENAI_API_KEY"] = "f2dab2d4f6fd4d738729e9d84573d6b5"
search = SerpAPIWrapper()
tools = [
    Tool(
        name="search",
        func=search.run,
        description="useful for when you need to answer questions about current events. You should ask targeted questions",
    ),
    WriteFileTool(),
    ReadFileTool(),
]

## Set up memory

The memory here is used for the agents intermediate steps

In [2]:
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

In [3]:
# Define your embedding model
import logging
import sys
logging.basicConfig(
    stream=sys.stdout, level=logging.INFO
)
from langchain.embeddings import HuggingFaceBgeEmbeddings

model_name = "bge-m3"
encode_kwargs = {
    'normalize_embeddings': True,  # 归一化向量以优化余弦相似度
    'query_instruction_for_retrieval': "为这个句子生成表示以用于检索相关文章："
}

In [4]:

from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceBgeEmbeddings

# 初始化模型
model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]
embeddings = model.encode(sentences)
similarities = model.similarity(embeddings, embeddings)
print(similarities)



/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:datasets:PyTorch version 2.6.0 available.
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]

tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [5]:

# Initialize the vectorstore as empty
import faiss
embedding_model = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",          # 模型名称
    model_kwargs={'device': 'cpu'},     # 运行设备（CPU/GPU）
    encode_kwargs={'normalize_embeddings': True}  # 归一化向量
)
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
# vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})
vectorstore = FAISS.from_texts(
    texts=["sunny", "windy"],
    embedding=embedding_model,  # 直接传入模型对象
    metadatas=[{"source": "1"}, {"source": "2"}]
)

# 检索示例
docs = vectorstore.similarity_search("query", k=3)

INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
INFO:faiss:Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes.
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-m3


/var/folders/lb/rklrmcf11s98sw5m_xzxjtcw0000gn/T/ipykernel_9588/3455455651.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceBgeEmbeddings(


## Setup model and AutoGPT

Initialize everything! We will use ChatOpenAI model

In [6]:
from langchain_experimental.autonomous_agents import AutoGPT
from langchain_openai import ChatOpenAI

In [18]:
from langchain_ollama import OllamaLLM
from langchain.llms.base import LLM
from typing import Any, List, Mapping, Optional
llm=OllamaLLM(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.7,
    num_ctx=2048,
    handle_parsing_errors=True,
    use_cache=True  # 启用缓存)  # 上下文长度)

)
# 初始化实例
agent = AutoGPT.from_llm_and_tools(
    ai_name="Tom",
    ai_role="Assistant",
    tools=tools,
    llm=llm,
    memory=vectorstore.as_retriever(),
)
# Set verbose to be true
agent.chain.verbose = True

## Run an example

Here we will make it write a weather report for SF

In [ ]:

from langchain.schema import Document
from langchain.chains.transform import TransformChain
from langchain_community.chat_message_histories import FileChatMessageHistory
from langchain.agents import initialize_agent
text="write a weather report for SF today"
text1="SF today is sunny in the morning and windy in the afternoon. The temperature is 20 degrees Celsius."
chunk_size = 1000  # 预留24 Token作为重叠防止语义断裂
# 正确：定义可调用的 transform 函数
def chunk_transform(inputs: dict) -> dict:
    text = inputs["input"]
    chunk_size = 500  # 分块大小
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    # 返回符合 output_variables 的字典
    return {"chunks": [Document(page_content=chunk) for chunk in chunks]}
chunk_chain = TransformChain(
    input_variables=["input"],
    output_variables=["chunks"],
    transform=chunk_transform
)
def debug_llm_call(input_text):
    print(f"Calling LLM with input: {input_text}")
    response = llm(input_text)
    print(f"LLM response: {response}")
    return response
tools = [
    Tool(
        name="process_long_text",
        func=lambda text: [
            # 关键修复点 1：正确调用链并访问 Document 内容
            print("input chunk:::", chunk.page_content)
            for chunk in chunk_chain.run({"input": text1})  # 关键修复点 2：输入字典格式
        ],
        description="用于处理超过模型限制的长文本输入"
    ),
    Tool(
        name="process_long_text",
        func=lambda text: "\n[继续]".join(
            # 关键修复点 1：正确调用链并访问 Document 内容
            llm(chunk.page_content)
            for chunk in chunk_chain.run({"input": text1})  # 关键修复点 2：输入字典格式
        ),
        description="用于处理超过模型限制的长文本输入"
    ),
    Tool(
        name="process_long_text",
        func=lambda text: debug_llm_call("\n".join(
            chunk.page_content
            for chunk in chunk_chain.run({"input": text1})
        )),
        description="用于处理超过模型限制的长文本输入"
    )
]



agent = initialize_agent(
    tools,
    llm,
    verbose=True,
    max_iterations=5  # 限制最大步数为 5
)# agent.run({"input":{"text":text}})
# Access the chat_history_memory attribute correctly

# Run the agent with the input text
aa = agent.run({"input": text})


AttributeError: 'AgentExecutor' object has no attribute 'chat_history_memory'

## Chat History Memory

In addition to the memory that holds the agent immediate steps, we also have a chat history memory. By default, the agent will use 'ChatMessageHistory' and it can be changed. This is useful when you want to use a different type of memory for example 'FileChatHistoryMemory'

In [ ]:
from langchain_community.chat_message_histories import FileChatMessageHistory

agent = AutoGPT.from_llm_and_tools(
    ai_name="Tom",
    ai_role="Assistant",
    tools=tools,
    llm=llm,
    memory=vectorstore.as_retriever(),
    chat_history_memory=FileChatMessageHistory("chat_history.txt"),
) #todo 为什么chat_history.txt是空的？
agent.run({"input": text})
# 确保在运行后保存历史

INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
{
    "thoughts": {
        "text": "Given the current constraints and goals, I should use the process_long_text command to input text that exceeds the 4000 word limit.",
        "reasoning": "I can process long texts by utilizing the process_long_text command with a json schema specifying the tool_input as a string.",
        "plan": "- Use process_long_text command\n- Input long text using the json schema",
        "criticism": "To avoid user assistance, I should focus on simple strategies without legal complications.",
        "speak": "I will use the process_long_text command to input the long text."
    },
    "command": {
        "name": "process_long_text",
        "args": {
            "tool_input": ""
        }
    }
}
Calling LLM with input: SF today is sunny in the morning and windy in the afternoon. The temperature is 20 degrees Celsius.
INFO:httpx:HTTP Request: POST http://localhost:11434/a

Token indices sequence length is longer than the specified maximum sequence length for this model (1320 > 1024). Running this sequence through the model will result in indexing errors


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
{
    "thoughts": {
        "text": "The current weather description is still not relevant to my task. I should focus on inputting text.",
        "reasoning": "I can process long texts regardless of external factors like weather.",
        "plan": "- Use process_long_text command\n- Input a new long text",
        "criticism": "I should avoid getting sidetracked by non-relevant information.",
        "speak": "Let's focus on the task at hand and get back to inputting text."
    },
    "command": {
        "name": "process_long_text",
        "args": {
            "tool_input": ""
        }
    }
}
Calling LLM with input: SF today is sunny in the morning and windy in the afternoon. The temperature is 20 degrees Celsius.
INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
LLM response: Sounds like it's going to be a bit of a mixed bag in San Francisco today!

In the mornin